# R11 — быстрый кешируемый OCR на GPU

PP-OCRv5 Mobile вместо медленной VL-модели. Обрабатываются все изображения, а результаты после каждого batch сохраняются в возобновляемый кеш на Google Drive. Сначала выполните smoke-тест и только затем включайте полный прогон.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys
import time

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/ecup')
INPUT_DIR = DRIVE_ROOT / 'input'
ARCHIVE_PATH = INPUT_DIR / 'images.zip'
MANIFEST_PATH = INPUT_DIR / 'image_manifest.parquet'
CACHE_DIR = DRIVE_ROOT / 'cache' / 'ocr_fast_v2'
OUTPUT_DIR = DRIVE_ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
assert ARCHIVE_PATH.exists(), f'Не найден {ARCHIVE_PATH}'
assert MANIFEST_PATH.exists(), f'Не найден {MANIFEST_PATH}'
print('Входные файлы найдены')

In [ ]:
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
REPO_URL = 'https://github.com/zimmer10/quality-control.git'
BRANCH = 'feature/R11-ocr-cache'
REPO_DIR = Path('/content/quality-control')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True).strip())

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'paddlepaddle-gpu==3.2.1',
    '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[ocr]'], cwd=REPO_DIR, check=True)
import paddle
assert paddle.is_compiled_with_cuda(), 'Установлен CPU PaddlePaddle вместо GPU-версии'
print('Paddle:', paddle.__version__)
print('GPU:', paddle.device.cuda.get_device_name())

In [ ]:
LOCAL_DATA = Path('/content/ecup_data')
IMAGES_ROOT = LOCAL_DATA / 'images'
EXTRACTED_MARKER = LOCAL_DATA / '.images_complete'
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
if not EXTRACTED_MARKER.exists():
    subprocess.run(['unzip', '-q', '-o', str(ARCHIVE_PATH), '-d', str(LOCAL_DATA)], check=True)
    assert IMAGES_ROOT.exists(), 'В архиве не найдена верхняя папка images/'
    EXTRACTED_MARKER.touch()
print('Изображения готовы:', IMAGES_ROOT)

In [ ]:
def run_ocr(limit, output_path, report_path):
    command = [
        sys.executable, '-m', 'ecup.features.ocr',
        '--manifest', str(MANIFEST_PATH),
        '--images-root', str(IMAGES_ROOT),
        '--cache-dir', str(CACHE_DIR),
        '--output', str(output_path),
        '--report', str(report_path),
        '--batch-size', '32',
        '--progress-every', '100',
        '--all-images',
    ]
    if limit is not None:
        command.extend(['--limit', str(limit)])
    environment = {
        **os.environ,
        'PYTHONPATH': str(REPO_DIR / 'src'),
        'CUDA_VISIBLE_DEVICES': '0',
    }
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO_DIR, env=environment, check=True)
    return time.perf_counter() - started

SMOKE_IMAGES = 100
seconds = run_ocr(
    SMOKE_IMAGES,
    OUTPUT_DIR / 'ocr_text_smoke_fast.parquet',
    OUTPUT_DIR / 'R11-ocr-smoke-fast.md',
)
seconds_per_image = seconds / SMOKE_IMAGES
estimated_hours = seconds_per_image * 49_456 / 3_600
print(f'{seconds_per_image:.3f} s/image; conservative full estimate: {estimated_hours:.2f} h')

In [ ]:
import polars as pl
smoke = pl.read_parquet(OUTPUT_DIR / 'ocr_text_smoke_fast.parquet')
display(smoke.group_by('ocr_status').len().sort('ocr_status'))
display(smoke.filter(pl.col('ocr_text_by_image') != '').select(
    'relative_path', 'ocr_quality', 'ocr_text_by_image'
).head(10))
assert smoke.height == SMOKE_IMAGES
assert smoke.filter(pl.col('ocr_status') == 'ocr_error').height == 0

## Полный прогон

Запускайте только после успешного smoke-теста. Поменяйте `RUN_FULL` на `True`. Уже готовые 100 изображений будут взяты из кеша.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    full_seconds = run_ocr(
        None,
        OUTPUT_DIR / 'ocr_text.parquet',
        OUTPUT_DIR / 'R11-ocr-cache.md',
    )
    full = pl.read_parquet(OUTPUT_DIR / 'ocr_text.parquet')
    display(full.group_by('ocr_status').len().sort('ocr_status'))
    print(f'Готово: {full.height} изображений за {full_seconds / 3_600:.2f} ч')
else:
    print('Полный прогон выключен. Сначала проверьте результаты smoke-теста.')